# 01 — FlowCast Exploratory Data Analysis (Northline Corridor)

Explores the 25-segment arterial corridor dataset (Jan–May 2025): diurnal volume curves, congestion distributions, weather degradation effects, and correlation structure.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

clean_df = pd.read_parquet('../data/interim/merged_clean.parquet')
clean_df['timestamp'] = pd.to_datetime(clean_df['timestamp'], utc=True)
print('Shape:', clean_df.shape)
clean_df.head()

In [ ]:
# Summary statistics across key numeric telemetry
clean_df[['traffic_volume', 'avg_speed', 'occupancy', 'travel_time', 'temperature', 'rainfall', 'visibility']].describe().round(2)

In [ ]:
# Congestion level distribution & accident base rate
print(clean_df['congestion_level'].value_counts(normalize=True).mul(100).round(2))
print('Accident positive window rate (%):', round((clean_df['accident_count'] > 0).mean() * 100, 3))

In [ ]:
# Weather impact on speed and incident frequency
weather_summary = clean_df.groupby('weather_condition').agg(
    windows=('traffic_volume', 'count'),
    avg_volume=('traffic_volume', 'mean'),
    avg_speed=('avg_speed', 'mean'),
    accident_rate_pct=('accident_count', lambda s: (s > 0).mean() * 100)
).round(2)
weather_summary

## Key EDA Takeaways Driving Feature Engineering
1. Strong bimodal weekday peaks (07:00–10:00 and 17:00–20:00) vs flatter weekend profiles motivate cyclical encodings (`hour_sin/cos`, `dow_sin/cos`) and 7-day seasonal lags (`t-336`).
2. Rain and Fog raise accident window probability by ~4× (~3.4% vs ~0.8% in Clear weather), justifying explicit weather × congestion interaction features.